In [ ]:
!pip install pyannote.audio
import torch
from pyannote.audio import Pipeline

# Ensure you have accepted the user conditions on the Hugging Face model page
# and generated a "Read" token at https://huggingface.co/settings/tokens
HF_TOKEN = "<token>"

# Initialize the pipeline
diarization_pipeline = Pipeline.from_pretrained(
    "pyannote/speaker-diarization-3.1",
    token=HF_TOKEN
)

def get_speaker_segments(audio_path):
    # Runs diarization and returns speaker intervals
    diarization = diarization_pipeline(audio_path)
    return diarization

# Example usage:
# segments = get_speaker_segments("clinical_consultation.wav")

config.yaml:   0%|          | 0.00/469 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/5.91M [00:00<?, ?B/s]

plda/xvec_transform.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

plda/plda.npz:   0%|          | 0.00/134k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/26.6M [00:00<?, ?B/s]

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Create a permanent workspace folder
workspace = "/content/drive/MyDrive/my_asr_project"
os.makedirs(workspace, exist_ok=True)

# Change directory to this folder so all relative paths (like './')
# now point to your Google Drive instead of the temporary /content/ folder
os.chdir(workspace)
print(f"Current working directory: {os.getcwd()}")

Mounted at /content/drive
Current working directory: /content/drive/MyDrive/my_asr_project


In [ ]:
!pip install torchao --upgrade
!pip install -q peft bitsandbytes
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig


# Load the base model
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Define LoRA configuration
# We target the Q and V projections in the encoder to manage attention behavior
peft_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

# Wrap the model with LoRA
model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647


In [ ]:
import torch.nn.functional as F

def decorrelation_loss(hidden_states, bos_embedding, threshold=0.3):
    """
    Penalizes attention collapse at language boundaries.
    hidden_states: Current token representations
    bos_embedding: Embedding of the start token
    """
    # Calculate cosine similarity between tokens and BOS
    cos_sim = F.cosine_similarity(hidden_states, bos_embedding, dim=-1)

    # Apply penalty only if similarity exceeds the threshold
    penalty = torch.clamp(cos_sim - threshold, min=0)

    return torch.sum(penalty)

In [ ]:
def normalize_medical_terms(transcribed_text, medical_kb):
    # This acts as your hallucination guard
    # Example: "sugar" -> "Diabetes Mellitus"
    for term, normalized in medical_kb.items():
        transcribed_text = transcribed_text.replace(term, normalized)
    return transcribed_text

In [ ]:
# Run this cell once
from huggingface_hub import login
login()

In [ ]:
from datasets import load_dataset, Audio, concatenate_datasets

# 1. Load WITHOUT streaming=True
# This downloads the metadata and the audio files locally.
print("Downloading data (this might take a moment)...")
ds_hindi = load_dataset("ai4bharat/Kathbath", "hindi", split="train[:100]")

# 3. Cast the audio column
# Now that files are on disk, this will successfully load the audio arrays.
combined_dataset = ds_hindi.cast_column("audio_filepath", Audio(sampling_rate=16000))

# 4. Verify
print("Sample 0 audio:", combined_dataset[0]["audio_filepath"])
# If this returns a dictionary with 'array' and 'sampling_rate', you are golden!

README.md:   0%|          | 0.00/10.0k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/22 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/32 [00:00<?, ?it/s]

hindi/valid-00000-of-00002.parquet:   0%|          | 0.00/258M [00:00<?, ?B/s]

hindi/valid-00001-of-00002.parquet:   0%|          | 0.00/318M [00:00<?, ?B/s]

...

hindi/train-00030-of-00032.parquet:   0%|          | 0.00/598M [00:00<?, ?B/s]

hindi/train-00031-of-00032.parquet:   0%|          | 0.00/603M [00:00<?, ?B/s]

Generating valid split:   0%|          | 0/3151 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/91752 [00:00<?, ? examples/s]

Sample 0 audio: <datasets.features._torchcodec.AudioDecoder object at 0x7c2616cdf7d0>


In [ ]:
# Check the features of the dataset to see if the column was actually cast
print(combined_dataset.features)

# Look at the raw data in the first row
row = combined_dataset[0]
print(f"Keys in row: {row.keys()}")

# Check the specific 'audio' column
audio_data = row["audio_filepath"]
print(f"Type of audio column: {type(audio_data)}")
print(f"Value of audio column: {audio_data}")

{'fname': Value('string'), 'text': Value('string'), 'audio_filepath': Audio(sampling_rate=16000, decode=True, stream_index=None), 'lang': Value('string'), 'duration': Value('float64'), 'gender': Value('string'), 'speaker_id': Value('int64')}
Keys in row: dict_keys(['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'])
Type of audio column: <class 'datasets.features._torchcodec.AudioDecoder'>
Value of audio column: <datasets.features._torchcodec.AudioDecoder object at 0x7c2616cddd00>


In [ ]:
print("New keys:", combined_dataset[0].keys())
print("Sample 0 audio:", combined_dataset[0]["audio_filepath"])
# Try to access the first item's audio array directly
sample = combined_dataset[0]
print("Audio key content:", sample["audio_filepath"])
# If this is {'array': array(...), 'path': ..., 'sampling_rate': ...}, it works!
# If this is None or just a path string, the cast failed.

New keys: dict_keys(['fname', 'text', 'audio_filepath', 'lang', 'duration', 'gender', 'speaker_id'])
Sample 0 audio: <datasets.features._torchcodec.AudioDecoder object at 0x7c2616cddfa0>
Audio key content: <datasets.features._torchcodec.AudioDecoder object at 0x7c2616cde9f0>


In [ ]:
from transformers import WhisperProcessor
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)
# 1. Force the feature extractor to use 80 channels
processor.feature_extractor.feature_size = 80
processor.feature_extractor.num_mel_bins = 80

# 2. Re-verify the configuration
print(f"Feature size: {processor.feature_extractor.feature_size}")
def prepare_dataset_final_v6(batch):
    try:
        # 1. Get the samples
        audio_samples = batch["audio_filepath"].get_all_samples()

        # 2. Extract the data tensor and convert to NumPy
        # audio_samples.data is the torch tensor
        # .squeeze() removes the extra [1, ...] dimension to make it [105883]
        speech = audio_samples.data.squeeze().numpy()

        # 3. Process with your processor
        batch["input_features"] = processor.feature_extractor(
            speech,
            sampling_rate=16000
        ).input_features[0]

        batch["labels"] = processor.tokenizer(batch["text"]).input_ids
        return batch
    except Exception as e:
        # If any step fails, filter it out
        return {"input_features": None, "labels": None}

# Run the mapping
processed_dataset = combined_dataset.map(prepare_dataset_final_v6)
processed_dataset = processed_dataset.filter(lambda x: x["input_features"] is not None)

print(f"Final Count: {len(processed_dataset)}")

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Feature size: 80


Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Filter:   0%|          | 0/100 [00:00<?, ? examples/s]

Final Count: 100


In [ ]:
# Define your permanent path
save_path = "/content/drive/MyDrive/processed_kathbath_dataset"

# Save the result of your filter operation
processed_dataset.save_to_disk(save_path)
print("Dataset saved permanently to Drive!")

NameError: name 'processed_dataset' is not defined

In [ ]:
# Test this line
print(combined_dataset[0]["audio_filepath"].get_all_samples())

AudioSamples:
  data (shape): torch.Size([1, 79877])
  pts_seconds: 0.0
  duration_seconds: 4.9923125
  sample_rate: 16000



In [ ]:
# The AudioDecoder object usually holds the path in a property.
# Let's inspect the object attributes.
audio_obj = combined_dataset[0]["audio_filepath"]

# Try to see if it has a 'path' attribute
if hasattr(audio_obj, 'path'):
    print(f"Found path: {audio_obj.path}")
else:
    # If not, let's print all attributes to find where the path is hidden
    print(f"Attributes: {dir(audio_obj)}")

Attributes: ['__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getitem__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_decoder', '_desired_sample_rate', '_hf_encoded', 'get_all_samples', 'get_samples_played_in_range', 'metadata', 'stream_index']


In [ ]:
from datasets import load_from_disk
# This loads your data instantly from Drive
processed_dataset = load_from_disk("/content/drive/MyDrive/processed_kathbath_dataset")
print(f"Dataset loaded. Total samples: {len(processed_dataset)}")

Dataset loaded. Total samples: 100


In [ ]:
from dataclasses import dataclass
from typing import Any, Dict, List, Union
import torch

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # Split inputs and labels
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        label_features = [{"input_ids": feature["labels"]} for feature in features]

        # Collate (pad) the inputs and labels
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Replace padding with -100 to ignore it in the loss function
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Remove BOS token from the beginning of labels if it's there
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels
        return batch

In [ ]:
from transformers import WhisperForConditionalGeneration

model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")

# Force the correct generation configuration to avoid warnings
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

config.json:   0%|          | 0.00/1.97k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/967M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.87k [00:00<?, ?B/s]

In [ ]:
from transformers import Seq2SeqTrainingArguments

# Define the training arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="/content/local_checkpoints",
    # 1. Performance boosts
    per_device_train_batch_size=1,         # Pushing to 2 for better GPU utilization
    gradient_accumulation_steps=32,        # Effective batch size= = 32 (balanced)

    # 2. Precision & Optimization (T4 must use fp16, not bf16)
    fp16=True,                             # Enable fp16 (T4 supported)
    bf16=False,                            # Disable bf16 (T4 unsupported)
    optim="adamw_8bit",                    # Keep this

    # 3. Learning & Steps
    learning_rate=5e-5,                    # Lowered slightly for stability at batch size 2
    warmup_steps=20,
    max_steps=200,                         # Kept low to ensure completion in < 2 hours

    # 4. Memory & Checkpointing
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": True},

    # 5. Speed optimization: Disable eval/generate during training
    eval_strategy="no",                    # Disable eval to save time and VRAM
    predict_with_generate=False,           # Massive speed boost: don't generate text during train

    # 6. Logging
    logging_steps=10,                      # More frequent feedback
    save_steps=100,                        # Save less often
    save_total_limit=1,
)

In [ ]:
# Filter out files longer than 20 seconds to prevent OOM
processed_dataset = processed_dataset.filter(lambda x: len(x["input_features"]) < 3000)

In [ ]:
!pip install torchao --upgrade
!pip install -q peft bitsandbytes
from transformers import WhisperForConditionalGeneration
from peft import get_peft_model, LoraConfig
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small")
peft_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
)

# Wrap the model with LoRA
model = get_peft_model(model, peft_config)
model.config.use_cache = False  # <--- CRITICAL

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 38.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 14.1 MB/s eta 0:00:00


Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
from transformers import WhisperProcessor
model_id = "openai/whisper-small"
processor = WhisperProcessor.from_pretrained(model_id)

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/283k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/836k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.48M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

PeftModel(
  (base_model): LoraModel(
    (model): WhisperForConditionalGeneration(
      (model): WhisperModel(
        (encoder): WhisperEncoder(
          (conv1): Conv1d(80, 768, kernel_size=(3,), stride=(1,), padding=(1,))
          (conv2): Conv1d(768, 768, kernel_size=(3,), stride=(2,), padding=(1,))
          (embed_positions): Embedding(1500, 768)
          (layers): ModuleList(
            (0-11): 12 x WhisperEncoderLayer(
              (self_attn): WhisperAttention(
                (k_proj): Linear(in_features=768, out_features=768, bias=False)
                (v_proj): lora.Linear(
                  (base_layer): Linear(in_features=768, out_features=768, bias=True)
                  (lora_dropout): ModuleDict(
                    (default): Dropout(p=0.05, inplace=False)
                  )
                  (lora_A): ModuleDict(
                    (default): Linear(in_features=768, out_features=8, bias=False)
                  )
                  (lora_B): ModuleDict(
   

In [ ]:
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor
from peft import get_peft_model, LoraConfig


# 1. Load your base model
model = AutoModelForSpeechSeq2Seq.from_pretrained("openai/whisper-small")

# 2. FIX: Enable input gradients (Must be done before adding LoRA)
model.enable_input_require_grads()

# 3. Apply LoRA configuration
peft_config = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj"],
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none"
) # Ensure your config is defined
model = get_peft_model(model, peft_config)

# 4. Verify that trainable parameters exist
model.print_trainable_parameters()

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

trainable params: 884,736 || all params: 242,619,648 || trainable%: 0.3647


In [ ]:
from transformers import Seq2SeqTrainer, TrainerCallback

# 1. Define the custom alert callback
class ProgressAlertCallback(TrainerCallback):
    def on_log(self, args, state, control, logs=None, **kwargs):
        # Alerts every 10 steps so you can monitor progress
        if state.global_step % 10 == 0:
            loss = logs.get("loss", "N/A")
            print(f"--- [PROGRESS ALERT] Step {state.global_step}: Training is moving smoothly! Current Loss: {loss} ---")

    def on_train_end(self, args, state, control, **kwargs):
        print("!!! TRAINING COMPLETE: The model has finished fine-tuning. Time to merge and save! !!!")

# 2. Initialize the trainer
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)
for param in model.parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=processed_dataset,
    # eval_dataset=None, # Removed to save VRAM and time since eval_strategy is "no"
    data_collator=data_collator,
    processing_class=processor,
)

# 3. Add the callback
trainer.add_callback(ProgressAlertCallback())
trainer.train()

Step,Training Loss
10,60.292743
20,51.179428
30,52.113495
40,40.717007
50,41.467365
60,34.671271
70,36.232062
80,30.490503
90,31.783707
100,26.371298


--- [PROGRESS ALERT] Step 10: Training is moving smoothly! Current Loss: 60.29274291992188 ---
--- [PROGRESS ALERT] Step 20: Training is moving smoothly! Current Loss: 51.17942810058594 ---
--- [PROGRESS ALERT] Step 30: Training is moving smoothly! Current Loss: 52.113494873046875 ---
--- [PROGRESS ALERT] Step 40: Training is moving smoothly! Current Loss: 40.71700744628906 ---
--- [PROGRESS ALERT] Step 50: Training is moving smoothly! Current Loss: 41.46736450195313 ---
--- [PROGRESS ALERT] Step 60: Training is moving smoothly! Current Loss: 34.671270751953124 ---
--- [PROGRESS ALERT] Step 70: Training is moving smoothly! Current Loss: 36.232061767578124 ---
--- [PROGRESS ALERT] Step 80: Training is moving smoothly! Current Loss: 30.4905029296875 ---
--- [PROGRESS ALERT] Step 90: Training is moving smoothly! Current Loss: 31.783706665039062 ---
--- [PROGRESS ALERT] Step 100: Training is moving smoothly! Current Loss: 26.371298217773436 ---
--- [PROGRESS ALERT] Step 110: Training is mo

Step,Training Loss
10,60.292743
20,51.179428
30,52.113495
40,40.717007
50,41.467365
60,34.671271
70,36.232062
80,30.490503
90,31.783707
100,26.371298


--- [PROGRESS ALERT] Step 140: Training is moving smoothly! Current Loss: 22.370085144042967 ---
--- [PROGRESS ALERT] Step 150: Training is moving smoothly! Current Loss: 24.166719055175783 ---
--- [PROGRESS ALERT] Step 160: Training is moving smoothly! Current Loss: 21.2349853515625 ---
--- [PROGRESS ALERT] Step 170: Training is moving smoothly! Current Loss: 23.2013671875 ---
--- [PROGRESS ALERT] Step 180: Training is moving smoothly! Current Loss: 20.180015563964844 ---
--- [PROGRESS ALERT] Step 190: Training is moving smoothly! Current Loss: 22.400054931640625 ---
--- [PROGRESS ALERT] Step 200: Training is moving smoothly! Current Loss: 19.880952453613283 ---
--- [PROGRESS ALERT] Step 200: Training is moving smoothly! Current Loss: N/A ---
!!! TRAINING COMPLETE: The model has finished fine-tuning. Time to merge and save! !!!


TrainOutput(global_step=200, training_loss=31.787543182373046, metrics={'train_runtime': 1909.9172, 'train_samples_per_second': 3.351, 'train_steps_per_second': 0.105, 'total_flos': 1.4492971008e+18, 'train_loss': 31.787543182373046, 'epoch': 50.0})

In [ ]:
import torch
print(torch.cuda.is_available()) # Must be True
print(torch.cuda.current_device()) # Should be 0

True
0


In [ ]:
import shutil

# Define where you want the model to live on Drive
drive_path = "/content/drive/MyDrive/my_asr_project/final_model"

# Copy the local checkpoints to Drive
shutil.copytree("/content/local_checkpoints", drive_path)
print(f"Model successfully saved to: {drive_path}")

Model successfully saved to: /content/drive/MyDrive/my_asr_project/final_model


In [ ]:
import os

# Define your drive path
drive_path = "/content/drive/MyDrive/my_asr_project/final_model"

# List all files inside the folder to verify
files = os.listdir(drive_path)
print(f"Files found in {drive_path}:")
for file in files:
    print(f"- {file}")

Files found in /content/drive/MyDrive/my_asr_project/final_model:
- checkpoint-200


In [ ]:
from datasets import load_dataset

# Load a specific split or just a subset to test your pipeline
# 'split="train[:100]"' loads only the first 100 examples
dataset = load_dataset("ekacare/eka-medical-asr-evaluation-dataset", split="test[:100]")

# Now you can inspect it
print(dataset[0])

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.
Some datasets params were ignored: ['default_preview_rows']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


en/test-00000.parquet:   0%|          | 0.00/23.0M [00:00<?, ?B/s]

en/test-00001.parquet:   0%|          | 0.00/14.2M [00:00<?, ?B/s]

en/test-00002.parquet:   0%|          | 0.00/14.4M [00:00<?, ?B/s]

...

en/test-00005.parquet:   0%|          | 0.00/60.2M [00:00<?, ?B/s]

en/test-00006.parquet:   0%|          | 0.00/63.1M [00:00<?, ?B/s]

en/test-00007.parquet:   0%|          | 0.00/15.8M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/3619 [00:00<?, ? examples/s]

{'md5_text': '05e1834395bd772db887a07769e53c3d', 'file_name': '/250427/ce-73d4588f-143b-4ffb-8944-a8df7f811c16/7.m4a', 'audio': <datasets.features._torchcodec.AudioDecoder object at 0x789406d8d340>, 'md5_audio': 'df654cc44ffd1fd350e960c4b3555573', 'duration': 10.496000289916992, 'text': 'not having adequate rest. Okay okay. So that continues for better part of the day. Hmm, hmm, hmm. Then', 'audio_language': 'en', 'text_language': 'en', 'session_id': 'ce-73d4588f-143b-4ffb-8944-a8df7f811c16', 'speaker': None, 'type_concept': 'misc_medical', 'recording_context': 'conversation', 'medical_entities': '[["adequate rest", "advices", [[11, 24]]]]'}


In [ ]:
print(f"Loaded {len(dataset)} evaluation samples.")

Loaded 100 evaluation samples.


In [ ]:
medical_scripts = [
    "Doctor: Hello, how are you feeling? Patient: I have been having a severe headache for three days. Doctor: I see, let's check your blood pressure; it is slightly high.",
    "Patient: Doctor, my back pain is worse today. Doctor: Have you been taking the Ibuprofen I prescribed? Patient: Only once, it made my stomach feel a bit uneasy.",
    "Doctor: Your blood sugar levels are slightly elevated. Patient: Should I start taking the Metformin again? Doctor: Yes, please take five hundred milligrams twice a day.",
    "Patient: I have a persistent cough and a fever. Doctor: Let me listen to your lungs. It sounds like a viral infection, drink plenty of fluids.",
    "Doctor: The report shows your cholesterol is high. Patient: Do I need to change my diet? Doctor: Yes, avoid oily foods and continue with the Atorvastatin.",
    "Patient: My skin has been very itchy lately. Doctor: This looks like an allergic reaction, I will prescribe an antihistamine called Cetirizine.",
    "Doctor: How is the pain in your knee? Patient: It is better, but it still hurts when I climb stairs. Doctor: Keep using the prescribed ointment.",
    "Patient: I feel very dizzy when I stand up. Doctor: That can happen with your current medication, let us adjust the dosage of your Amlodipine.",
    "Doctor: Did you complete the course of antibiotics? Patient: Yes, the swelling in my throat has gone down significantly.",
    "Patient: I have been feeling very anxious at night. Doctor: Let us try a mild sedative like Escitalopram for two weeks.",
    "Doctor: Your thyroid levels are a bit off today. Patient: Should I increase my thyroxine dose? Doctor: Yes, take seventy-five micrograms from tomorrow.",
    "Patient: My eyes have been watering a lot since morning. Doctor: That is common with seasonal allergies, use these eye drops three times a day.",
    "Doctor: How are you managing your diabetes? Patient: I am trying to walk more, but my feet still feel numb. Doctor: We need to monitor your nerve health more closely.",
    "Patient: I have a sharp pain in my chest. Doctor: Please sit down; I need to perform an EKG immediately to rule out any cardiac issues.",
    "Doctor: Your iron levels are quite low. Patient: Is that why I feel so tired? Doctor: Yes, take these iron supplements after your lunch daily.",
    "Patient: I have a rash on my arm. Doctor: That looks like contact dermatitis, apply this hydrocortisone cream twice daily.",
    "Doctor: Have you had any nausea with the new medication? Patient: A little bit, but it goes away after I eat something.",
    "Patient: My child has been vomiting since last night. Doctor: Let us check for dehydration; give them some ORS in small sips.",
    "Doctor: Your blood pressure is much better today. Patient: I think the change in diet really helped. Doctor: Excellent, keep that up.",
    "Patient: I am worried about this persistent cough. Doctor: Let us get a chest X-ray done to ensure there is no congestion in the lungs."
]

In [ ]:
!pip install kokoro soundfile
# You also need espeak-ng for pronunciation
!apt-get install -y espeak-ng

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 1.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 30.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 51.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.5/163.5 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.7/82.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 237.9/237.9 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 734.0/734.0 kB 18.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.7/216.7 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 31.2 MB/s eta 0:00:00


In [ ]:
from kokoro import KPipeline
import soundfile as sf
import os
import json
from google.colab import drive
import numpy as np

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define the path in your Drive where you want to save everything
# Change 'Medical_Audio_Dataset' to your preferred folder name
save_directory = "/content/drive/MyDrive/Medical_Audio_Dataset"
os.makedirs(save_directory, exist_ok=True)

# 3. Initialize Pipeline
pipeline = KPipeline(lang_code='a')
voice = 'af_heart'
prepared_data = []

# 4. Process scripts
for i, script in enumerate(medical_scripts):
    # Path now points to your Drive directly
    output_path = os.path.join(save_directory, f"sample_{i}.wav")

    # Generate and concatenate audio
    generator = pipeline(script, voice=voice, speed=1.0)
    audio_segments = [audio for _, _, audio in generator]
    combined_audio = np.concatenate(audio_segments)

    # Save to file
    sf.write(output_path, combined_audio, 24000)

    # Append to list (storing the relative filename)
    prepared_data.append({"audio": f"sample_{i}.wav", "text": script})
    print(f"Synthesized: {output_path}")

# 5. Save the metadata JSON to the same folder
metadata_path = os.path.join(save_directory, "metadata.json")
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(prepared_data, f, indent=4)

print("-" * 30)
print(f"Batch generation complete!")
print(f"Audio files and metadata.json are saved in: {save_directory}")

Mounted at /content/drive
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_0.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_1.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_2.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_3.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_4.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_5.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_6.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_7.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_8.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_9.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_10.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_11.wav
Synthesized: /content/drive/MyDrive/Medical_Audio_Dataset/sample_12.wav
Synthesized: /content/drive/MyDrive/Medical_Audi